# 02 — Layer 2: Standardize (Synthea)

🔧 Script · 📚 Reference

Layer 2 converts every source into profile-validated FHIR R4 silver. For Synthea the data is already FHIR — the work is annotation: source-tag, lifecycle, and USCDI profile URLs. This notebook traces a single resource from bronze to silver.


## 1. Silver tier layout

In [ ]:
from pathlib import Path
import json

ATLAS_ROOT = Path("..").resolve()
CORPUS     = ATLAS_ROOT / "corpus"

silver_dir = CORPUS / "silver" / "synthea" / "rhett759"
print("Silver files:", [f.name for f in silver_dir.iterdir()])


## 2. Run SyntheaStandardizer

In [ ]:
# 🔧 Script + 📚 Reference (USCDI profiles)
from ehi_atlas.standardize.synthea import SyntheaStandardizer

standardizer = SyntheaStandardizer(
    bronze_root=CORPUS / "bronze",
    silver_root=CORPUS / "silver",
)
result = standardizer.standardize("rhett759")

print("Hash:              ", result.sha256[:16], "...")
print("Silver path:       ", result.silver_path.split("/")[-2:])
print("Validator errors:  ", len(result.validation_errors))
print("Validator warnings:", len(result.validation_warnings))


## 3. Bronze → silver: before and after

In [ ]:
# Pick a Condition entry from bronze and silver and compare meta
bronze_bundle = json.loads(
    (CORPUS / "bronze" / "synthea" / "rhett759" / "data.json").read_text()
)
silver_bundle = json.loads(
    (CORPUS / "silver" / "synthea" / "rhett759" / "bundle.json").read_text()
)

bronze_cond = next(
    e["resource"] for e in bronze_bundle["entry"]
    if e["resource"]["resourceType"] == "Condition"
)
silver_cond = next(
    e["resource"] for e in silver_bundle["entry"]
    if e["resource"]["resourceType"] == "Condition"
)

print("--- BRONZE meta.tag ---")
print(bronze_cond.get("meta", {}).get("tag", "(none)"))

print()
print("--- SILVER meta.tag ---")
for tag in silver_cond.get("meta", {}).get("tag", []):
    print(" ", tag)

print()
print("--- SILVER meta.profile ---")
for prof in silver_cond.get("meta", {}).get("profile", []):
    print(" ", prof)


## 4. BundleValidator output

In [ ]:
# 📚 Reference — validates against 32 known USCDI + CARIN-BB profile URLs
# BundleValidator.validate() returns a list of issue strings (empty = clean)
from ehi_atlas.standardize.validators import BundleValidator

validator = BundleValidator(strict=False)
issues = validator.validate(silver_bundle)

print(f"Validation issues: {len(issues)}")
if issues:
    print("First issue:", issues[0])
else:
    print("Silver bundle passes BundleValidator (non-strict mode).")


**Next:** [03_layer2b_vision_extraction.ipynb](./03_layer2b_vision_extraction.ipynb)